# DeepSORT Modernization Project

Clean Colab notebook for the final submission.

This notebook reproduces the main pipeline:

1. clone repository,
2. install dependencies,
3. download dataset,
4. run original DeepSORT baseline,
5. run YOLO11n detections,
6. extract OSNet x0.5 ReID features,
7. run final best DeepSORT configuration,
8. generate 12 overlay videos,
9. display original and best overlays for visual comparison.

Final selected configuration:

- detector: YOLO11n
- confidence threshold: 0.40
- detector IoU threshold: 0.60
- ReID model: OSNet x0.5
- `max_cosine_distance`: 0.30
- `nn_budget`: 100


## 1. Clone repository

In [ ]:
%cd /content
!rm -rf /content/deepsort-modern-project
!git clone https://github.com/Dokirma/deepsort-modern-project.git /content/deepsort-modern-project
%cd /content/deepsort-modern-project

!pwd
!ls


## 2. Install dependencies

In [ ]:
%cd /content/deepsort-modern-project

!pip install -q -r requirements.txt
!pip install -q ultralytics torchreid gdown


## 3. Download and extract dataset

In [ ]:
%cd /content/deepsort-modern-project

!python src/data/download_data.py \
  --file-id 1ujjjDlQZ6eEfdfWqJx-L_pgbJkSqRkU8 \
  --output data/raw/dlcv-final-project-videos.tar.gz \
  --extract-to data/mot

!find data/mot/videos -maxdepth 1 -type d | sort


## 4. Run original DeepSORT baseline on all six sequences

In [ ]:
%cd /content/deepsort-modern-project

!python src/tracking/run_original_deepsort_all.py

!echo "Original tracks:"
!ls -lh outputs/tracks/original_deepsort
!wc -l outputs/tracks/original_deepsort/*.txt


## 5. Run YOLO11n detections on all six sequences

In [ ]:
%cd /content/deepsort-modern-project

!python src/detectors/run_yolo_all.py \
  --model yolo11n.pt \
  --conf 0.40 \
  --iou 0.60 \
  --imgsz 640 \
  --name yolo11n_conf040

!echo "YOLO detections:"
!ls -lh outputs/detections/yolo11n_conf040


## 6. Extract OSNet x0.5 ReID features

In [ ]:
%cd /content/deepsort-modern-project

!python src/reid/run_osnet_features_all.py \
  --det-dir outputs/detections/yolo11n_conf040 \
  --output-dir outputs/detections_reid/yolo11n_osnet_x0_5 \
  --model osnet_x0_5 \
  --batch-size 32

!echo "OSNet x0.5 features:"
!ls -lh outputs/detections_reid/yolo11n_osnet_x0_5


## 7. Run final best DeepSORT configuration

In [ ]:
%cd /content/deepsort-modern-project

!python src/tracking/run_deepsort_from_reid_all.py \
  --reid-dir outputs/detections_reid/yolo11n_osnet_x0_5 \
  --tracker-name deepsort_yolo11n_osnet_x0_5_cos030 \
  --max-cosine-distance 0.30 \
  --nn-budget 100

!echo "Best tracks:"
!ls -lh outputs/tracks/deepsort_yolo11n_osnet_x0_5_cos030
!wc -l outputs/tracks/deepsort_yolo11n_osnet_x0_5_cos030/*.txt


## 8. Verify required tracking outputs

In [ ]:
%cd /content/deepsort-modern-project

import os

sequences = [
    "KITTI-17",
    "MOT16-09",
    "MOT16-11",
    "PETS09-S2L1",
    "TUD-Campus",
    "TUD-Stadtmitte",
]

all_ok = True

for seq in sequences:
    original_track = f"outputs/tracks/original_deepsort/{seq}.txt"
    best_track = f"outputs/tracks/deepsort_yolo11n_osnet_x0_5_cos030/{seq}.txt"
    ok_original = os.path.exists(original_track)
    ok_best = os.path.exists(best_track)
    print(seq, "original:", ok_original, "| best:", ok_best)
    all_ok = all_ok and ok_original and ok_best

print("READY FOR 12 OVERLAYS:", all_ok)


## 9. Generate 12 overlay videos

This creates:

- 6 original DeepSORT overlay videos,
- 6 final best modified DeepSORT overlay videos.


In [ ]:
%cd /content/deepsort-modern-project

import os
import subprocess

sequences = [
    "KITTI-17",
    "MOT16-09",
    "MOT16-11",
    "PETS09-S2L1",
    "TUD-Campus",
    "TUD-Stadtmitte",
]

original_video_dir = "outputs/videos/original_deepsort"
best_video_dir = "outputs/videos/best_yolo11n_osnet_x0_5"

os.makedirs(original_video_dir, exist_ok=True)
os.makedirs(best_video_dir, exist_ok=True)

for seq in sequences:
    print("=" * 80)
    print(f"Creating ORIGINAL overlay for {seq}")
    subprocess.run([
        "python", "src/visualization/make_overlay.py",
        "--sequence-dir", f"data/mot/videos/{seq}",
        "--tracks-file", f"outputs/tracks/original_deepsort/{seq}.txt",
        "--output-video", f"{original_video_dir}/{seq}_original_deepsort.mp4",
    ], check=True)

    print(f"Creating BEST overlay for {seq}")
    subprocess.run([
        "python", "src/visualization/make_overlay.py",
        "--sequence-dir", f"data/mot/videos/{seq}",
        "--tracks-file", f"outputs/tracks/deepsort_yolo11n_osnet_x0_5_cos030/{seq}.txt",
        "--output-video", f"{best_video_dir}/{seq}_best_yolo11n_osnet_x0_5.mp4",
    ], check=True)

print("Created 12 overlay videos.")


## 10. Convert overlay videos to browser-compatible H.264

In [ ]:
%cd /content/deepsort-modern-project

!mkdir -p outputs/videos_web/original_deepsort
!mkdir -p outputs/videos_web/best_yolo11n_osnet_x0_5

!for f in outputs/videos/original_deepsort/*.mp4; do \
  base=$(basename "$f"); \
  ffmpeg -y -i "$f" \
    -vcodec libx264 -pix_fmt yuv420p -movflags +faststart \
    -preset veryfast -crf 28 \
    "outputs/videos_web/original_deepsort/$base"; \
done

!for f in outputs/videos/best_yolo11n_osnet_x0_5/*.mp4; do \
  base=$(basename "$f"); \
  ffmpeg -y -i "$f" \
    -vcodec libx264 -pix_fmt yuv420p -movflags +faststart \
    -preset veryfast -crf 28 \
    "outputs/videos_web/best_yolo11n_osnet_x0_5/$base"; \
done


## 11. Verify 12 web-compatible videos

In [ ]:
%cd /content/deepsort-modern-project

!echo "COUNT:"
!find outputs/videos_web/original_deepsort outputs/videos_web/best_yolo11n_osnet_x0_5 -name "*.mp4" | wc -l

!echo ""
!echo "ORIGINAL WEB:"
!ls -lh outputs/videos_web/original_deepsort/*.mp4

!echo ""
!echo "BEST WEB:"
!ls -lh outputs/videos_web/best_yolo11n_osnet_x0_5/*.mp4


## 12. Links to all overlay videos

In [ ]:
%cd /content/deepsort-modern-project

from IPython.display import HTML, display

sequences = [
    "KITTI-17",
    "MOT16-09",
    "MOT16-11",
    "PETS09-S2L1",
    "TUD-Campus",
    "TUD-Stadtmitte",
]

rows = []

for seq in sequences:
    original_path = f"outputs/videos_web/original_deepsort/{seq}_original_deepsort.mp4"
    best_path = f"outputs/videos_web/best_yolo11n_osnet_x0_5/{seq}_best_yolo11n_osnet_x0_5.mp4"
    rows.append(
        "<tr>"
        f"<td>{seq}</td>"
        f"<td><a href='{original_path}' target='_blank'>Original overlay</a></td>"
        f"<td><a href='{best_path}' target='_blank'>Best overlay</a></td>"
        "</tr>"
    )

html = (
    "<h3>Web-compatible overlay videos for all six sequences</h3>"
    "<table border='1' style='border-collapse: collapse;'>"
    "<tr><th>Sequence</th><th>Original DeepSORT</th><th>Best modified DeepSORT</th></tr>"
    + "".join(rows)
    + "</table>"
)

display(HTML(html))


## 13. Display original and best overlays for TUD-Campus

In [ ]:
%cd /content/deepsort-modern-project

from IPython.display import Video, display

print("Original DeepSORT overlay: TUD-Campus")
display(Video(
    "outputs/videos_web/original_deepsort/TUD-Campus_original_deepsort.mp4",
    embed=True,
    width=700
))


In [ ]:
%cd /content/deepsort-modern-project

from IPython.display import Video, display

print("Best modified DeepSORT overlay: TUD-Campus")
display(Video(
    "outputs/videos_web/best_yolo11n_osnet_x0_5/TUD-Campus_best_yolo11n_osnet_x0_5.mp4",
    embed=True,
    width=700
))


## 14. Optional backup to Google Drive

This cell is optional. It saves generated outputs and videos to the current Google Drive account.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
from datetime import datetime

drive.mount("/content/drive", force_remount=True)

PROJECT = Path("/content/deepsort-modern-project")
BACKUP_DIR = Path("/content/drive/MyDrive/deepsort_submission_backup")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_with_videos = PROJECT / f"final_outputs_with_videos_{stamp}.zip"

items = [
    "outputs/tracks",
    "outputs/detections",
    "outputs/detections_reid",
    "outputs/videos_web",
    "report",
    "README.md",
]

cmd = ["zip", "-r", "-q", str(zip_with_videos)] + [i for i in items if (PROJECT / i).exists()]
subprocess.run(cmd, check=True)

dst = BACKUP_DIR / zip_with_videos.name
shutil.copy2(zip_with_videos, dst)

print("Backup saved to:", dst)
